# Multi-Demo Analysis — Keyframe Extractor Consistency

Runs all four extractors across every demo in one HDF5 file and produces
a consistency report: per-extractor distributions, summary statistics,
and outlier flags.

In [ ]:
# Cell 1 — Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.utils.loader import load_libero_demo, list_demos
from src.extractors import (
    UniformExtractor,
    VelocityZeroExtractor,
    GripperStateExtractor,
    AWEExtractor,
)
from src.extractors.base import KeyframeExtractor

DEMO_PATH = "~/keyframe_selection/LIBERO/libero/datasets/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_demo.hdf5"
RESULTS   = Path.cwd().parent / "results"
RESULTS.mkdir(exist_ok=True)

uniform  = UniformExtractor(n_keyframes=10)
velocity = VelocityZeroExtractor()           # adaptive p25
gripper  = GripperStateExtractor(min_dist=3)
awe      = AWEExtractor(error_threshold=0.01)

# Ordered list of (display_label, extractor, input_key)
# input_key refers to the field in the demo dict to pass to extract()
EXTRACTORS = [
    ("uniform",   uniform,  "ee_pos"),
    ("velocity",  velocity, "ee_vel"),
    ("gripper",   gripper,  "gripper_state"),
    ("awe",       awe,      "ee_pos"),
]

print("Extractors:")
for label, ext, key in EXTRACTORS:
    print(f"  {label:<10}  {ext.name}  (input: demo['{key}'])")

In [ ]:
# Cell 2 — Run all extractors on all demos
demos    = list_demos(DEMO_PATH)
n_demos  = len(demos)
print(f"Processing {n_demos} demos …")

rows = []
for demo_idx in range(n_demos):
    demo = load_libero_demo(DEMO_PATH, demo_idx=demo_idx)
    T    = len(demo["ee_pos"])

    for label, ext, input_key in EXTRACTORS:
        kf     = ext.extract(demo[input_key])
        cr     = KeyframeExtractor.compression_ratio(T, len(kf))
        thresh = getattr(ext, "threshold_used", float("nan"))
        rows.append({
            "demo_idx":         demo_idx,
            "extractor":        label,
            "n_keyframes":      len(kf),
            "compression_ratio": cr,
            "threshold_used":   thresh,
            "T":                T,
        })

    if (demo_idx + 1) % 10 == 0:
        print(f"  {demo_idx + 1}/{n_demos}")

df = pd.DataFrame(rows)
print(f"\nDataFrame shape: {df.shape}")
print(df.head(8).to_string(index=False))

In [ ]:
# Cell 3 — Summary statistics per extractor
LABEL_ORDER = ["uniform", "velocity", "gripper", "awe"]

print("=" * 70)
print("n_keyframes — mean / std / min / max")
print("=" * 70)
stats = (
    df.groupby("extractor", sort=False)["n_keyframes"]
    .agg(["mean", "std", "min", "max"])
    .reindex(LABEL_ORDER)
)
print(stats.round(2).to_string())

print()
print("=" * 70)
print("compression_ratio — mean / std / min / max")
print("=" * 70)
cr_stats = (
    df.groupby("extractor", sort=False)["compression_ratio"]
    .agg(["mean", "std", "min", "max"])
    .reindex(LABEL_ORDER)
)
print(cr_stats.round(4).to_string())

print()
print("=" * 70)
print("velocity: threshold_used — mean / std / min / max")
print("=" * 70)
vel_df = df[df["extractor"] == "velocity"]
t_stats = vel_df["threshold_used"].agg(["mean", "std", "min", "max"])
print(t_stats.round(6).to_string())

In [ ]:
# Cell 4 — Distribution plots
COLORS = {
    "uniform":  "tab:blue",
    "velocity": "tab:red",
    "gripper":  "tab:green",
    "awe":      "tab:orange",
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
fig.suptitle(
    f"Keyframe count distributions across {n_demos} demos\n"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate",
    fontsize=10,
)

for ax, label in zip(axes, LABEL_ORDER):
    subset = df[df["extractor"] == label]["n_keyframes"]
    mean   = subset.mean()

    # Use integer bins spanning the full observed range
    lo, hi  = int(subset.min()), int(subset.max())
    bins    = range(lo, hi + 2)   # +2 so last bar is fully drawn

    ax.hist(subset, bins=bins, color=COLORS[label], alpha=0.75,
            edgecolor="white", linewidth=0.6, align="left")
    ax.axvline(mean, color="black", linewidth=1.4, linestyle="--",
               label=f"mean={mean:.1f}")
    ax.set_title(f"{label}\nmean={mean:.1f} keyframes", fontsize=10)
    ax.set_xlabel("n_keyframes", fontsize=9)
    ax.set_ylabel("demo count", fontsize=9)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(RESULTS / "multi_demo_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/multi_demo_distributions.png")

In [ ]:
# Cell 5 — Flag outliers
LOW, HIGH = 3, 20
outliers = df[(df["n_keyframes"] < LOW) | (df["n_keyframes"] > HIGH)].copy()

if outliers.empty:
    print(f"No outliers found (all extractors produced {LOW}–{HIGH} keyframes on every demo).")
else:
    print(f"{len(outliers)} outlier rows (n_keyframes < {LOW} or > {HIGH}):")
    print()
    print(outliers[["demo_idx", "extractor", "n_keyframes", "compression_ratio", "T"]]
          .sort_values(["extractor", "demo_idx"])
          .to_string(index=False))
    print()
    print("Per-extractor outlier counts:")
    print(outliers.groupby("extractor")["demo_idx"].count().reindex(LABEL_ORDER).fillna(0).astype(int).to_string())